# 02 · Spark: flights with UTC timestamps

**What this notebook does**
1. Start a local Spark session and read the monthly Parquet files from `00_download_bts.ipynb`.
2. Give every airport an IANA time zone (from its coordinates).
3. Convert BTS local times to **UTC**: `sched_dep_utc`, `dep_utc`, `sched_arr_utc`, `arr_utc`.
4. Verify (daylight saving, row counts, a number computed in Spark and in DuckDB).
5. Write the result as Parquet (`data/flights_utc`, partitioned by year) for the next notebook (delay chains + weather join).

**Why UTC is needed:** BTS times are *local* with no time zone. Ordering an aircraft's flights, or joining weather by hour, is wrong across time zones, daylight-saving changes, and midnight unless everything is in one clock.

**Trick used:** only the origin's time zone is needed. Actual departure = scheduled departure + `DepDelay`; scheduled arrival = scheduled departure + `CRSElapsedTime`; actual arrival = scheduled arrival + `ArrDelay` (all minutes).

In [16]:
import os, urllib.request
from pathlib import Path
import duckdb, pandas as pd
from pyspark.sql import SparkSession, functions as F

In [17]:
PROJECT = Path(os.path.expanduser("~/Desktop/ADSP32030_Scalable/Project"))
FLIGHTS_PQ = PROJECT/"data/flights/parquet"
OUT = PROJECT/"data/flights_utc"
META = PROJECT/"data/meta"; META.mkdir(parents=True, exist_ok=True)
AIRPORTS_CSV = PROJECT/"data/weather/meta/airports.csv"      # downloaded by 00_download_weather.ipynb
AIRPORTS_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"

spark = (SparkSession.builder.appName("flights-utc").master("local[*]")
         .config("spark.driver.memory", "8g")
         .config("spark.driver.bindAddress", "127.0.0.1")    # local mode: avoids hostname/VPN problems
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.session.timeZone", "UTC")        # all timestamps are UTC
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism, "| UI: http://localhost:4040")

Spark 4.3.0 | cores: 8 | UI: http://localhost:4040


## 1 · Read flights

In [18]:
flights = spark.read.parquet(str(FLIGHTS_PQ))        # all monthly files, 2010-2026
n_spark = flights.count()
print(f"{n_spark:,} flights in Spark, {len(flights.columns)} columns")
print("input partitions:", flights.rdd.getNumPartitions())

105,631,230 flights in Spark, 37 columns
input partitions: 22


## 2 · Time zone for every airport (from coordinates)

In [19]:
from timezonefinder import TimezoneFinder
if not AIRPORTS_CSV.exists():
    AIRPORTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(AIRPORTS_URL, AIRPORTS_CSV)

codes = [r[0] for r in flights.select("Origin").distinct().collect()]
oa = pd.read_csv(AIRPORTS_CSV, low_memory=False)
oa = oa[oa.iata_code.isin(codes)].sort_values("type", key=lambda s: s.map({"large_airport":0,"medium_airport":1}).fillna(2))
oa = oa.drop_duplicates("iata_code")[["iata_code","latitude_deg","longitude_deg"]]
tf = TimezoneFinder()
oa["tz"] = [tf.timezone_at(lat=a, lng=b) for a, b in zip(oa.latitude_deg, oa.longitude_deg)]
tz = oa.rename(columns={"iata_code":"Origin"})[["Origin","tz"]].dropna()
# manual overrides for airports the coordinate lookup missed (PBI is open; ISN and PFN are closed airports)
OVERRIDES = {"PBI": "America/New_York", "ISN": "America/Chicago", "PFN": "America/Chicago"}
tz = (pd.concat([tz, pd.DataFrame({"Origin": list(OVERRIDES), "tz": list(OVERRIDES.values())})])
        .drop_duplicates("Origin", keep="last"))
tz.to_csv(META/"airport_tz.csv", index=False)
missing = sorted(set(codes) - set(tz.Origin))
print(len(tz), "airports with a time zone; without one:", missing or "none")
tz.sample(min(5, len(tz)), random_state=1)

410 airports with a time zone; without one: none


,Origin,tz
38810,ACK,America/New_York
42798,SLC,America/Denver
42791,SJT,America/Chicago
38940,AVP,America/New_York
38927,ATW,America/Chicago


In [20]:
# how many FLIGHTS do the airports without a time zone account for?
if missing:
    miss_n = flights.filter(F.col("Origin").isin(missing)).count()
    print(f"flights from airports with no time zone: {miss_n:,} ({100*miss_n/n_spark:.3f}%)  -> these get null UTC times")

## 3 · Local → UTC

In [21]:
tz_sdf = F.broadcast(spark.createDataFrame(tz))          # ~400 rows: broadcast join, no shuffle

def hhmm_seconds(col):                                   # 1530 -> 55800 s; 2400 -> 86400 s (= next midnight)
    return (F.floor(col/100)*3600 + (col % 100)*60)

local_midnight = F.unix_timestamp(F.col("FlightDate").cast("timestamp"))     # session tz is UTC, so this is naive 00:00
naive_local_dep = F.timestamp_seconds(local_midnight + hhmm_seconds(F.col("CRSDepTime")))

def plus_minutes(ts, minutes):
    return F.timestamp_seconds(F.unix_timestamp(ts) + minutes*60)

fu = (flights.join(tz_sdf, "Origin", "left")
      .withColumn("sched_dep_utc", F.to_utc_timestamp(naive_local_dep, F.col("tz")))
      .withColumn("dep_utc",       plus_minutes(F.col("sched_dep_utc"), F.col("DepDelay")))
      .withColumn("sched_arr_utc", plus_minutes(F.col("sched_dep_utc"), F.col("CRSElapsedTime")))
      .withColumn("arr_utc",       plus_minutes(F.col("sched_arr_utc"), F.col("ArrDelay")))
      .withColumn("Year", F.col("Year").cast("int"))
      .select("Year","FlightDate","Reporting_Airline","Tail_Number","Flight_Number_Reporting_Airline",
              "Origin","Dest","tz","sched_dep_utc","dep_utc","sched_arr_utc","arr_utc",
              "CRSDepTime","CRSElapsedTime","DepDelay","ArrDelay","DepDel15","ArrDel15","TaxiOut","TaxiIn","Distance",
              "Cancelled","Diverted","CancellationCode","CarrierDelay","WeatherDelay","NASDelay",
              "SecurityDelay","LateAircraftDelay")
      .withColumnRenamed("tz", "origin_tz"))
fu.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: long (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- origin_tz: string (nullable = true)
 |-- sched_dep_utc: timestamp (nullable = true)
 |-- dep_utc: timestamp (nullable = true)
 |-- sched_arr_utc: timestamp (nullable = true)
 |-- arr_utc: timestamp (nullable = true)
 |-- CRSDepTime: double (nullable = true)
 |-- CRSElapsedTime: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- TaxiOut: double (nullable = true)
 |-- TaxiIn: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- Diverted: double (nullable = true)
 |-- CancellationCode: s

## 4 · Checks

In [22]:
# (a) Daylight saving check, measured directly: UTC minus local clock time (hours) must be constant within a month.
#     Expected: ORD 6 (Jan) / 5 (Jul), LAX 8 / 7, PHX 7 / 7, HNL 10 / 10  -> min_h == max_h in every row.
naive = F.timestamp_seconds(F.unix_timestamp(F.col("FlightDate").cast("timestamp")) + hhmm_seconds(F.col("CRSDepTime")))
off = (fu.filter((F.col("Year") == 2024) & F.col("Origin").isin("ORD","LAX","PHX","HNL") & F.col("sched_dep_utc").isNotNull())
         .withColumn("naive_local", naive)
         .withColumn("utc_minus_local_h", (F.unix_timestamp("sched_dep_utc") - F.unix_timestamp("naive_local")) / 3600)
         .withColumn("m", F.month("FlightDate")))
(off.filter(F.col("m").isin(1, 7)).groupBy("Origin", "m")
    .agg(F.count("*").alias("n"), F.min("utc_minus_local_h").alias("min_h"), F.max("utc_minus_local_h").alias("max_h"))
    .orderBy("Origin", "m").show())

+------+---+-----+-----+-----+
|Origin|  m|    n|min_h|max_h|
+------+---+-----+-----+-----+
|   HNL|  1| 5169| 10.0| 10.0|
|   HNL|  7| 5494| 10.0| 10.0|
|   LAX|  1|15228|  8.0|  8.0|
|   LAX|  7|17518|  7.0|  7.0|
|   ORD|  1|20321|  6.0|  6.0|
|   ORD|  7|24531|  5.0|  5.0|
|   PHX|  1|15378|  7.0|  7.0|
|   PHX|  7|15733|  7.0|  7.0|
+------+---+-----+-----+-----+



In [23]:
# (b) null UTC times and impossible durations
stats = fu.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("sched_dep_utc").isNull().cast("int")).alias("null_sched_dep_utc"),
    F.sum(F.col("sched_arr_utc").isNull().cast("int")).alias("null_sched_arr_utc"),
    F.sum(((F.col("sched_arr_utc") <= F.col("sched_dep_utc"))).cast("int")).alias("arr_not_after_dep"),
    F.sum(((F.col("arr_utc") < F.col("dep_utc"))).cast("int")).alias("actual_arr_before_dep"),
).toPandas()
stats

,rows,null_sched_dep_utc,null_sched_arr_utc,arr_not_after_dep,actual_arr_before_dep
0,105631230,1,212,29,34


In [24]:
# (c) ask it twice: same numbers from Spark and from DuckDB
YEAR = 2024
sp = fu.filter(F.col("Year") == YEAR).agg(F.count("*").alias("n"), F.avg("DepDelay").alias("avg_dep_delay")).toPandas()
dk = duckdb.connect().execute(f"SELECT count(*) AS n, avg(DepDelay) AS avg_dep_delay FROM read_parquet('{FLIGHTS_PQ}/*.parquet') WHERE Year = {YEAR}").df()
print("Spark :", sp.iloc[0].to_dict()); print("DuckDB:", dk.iloc[0].to_dict())
assert int(sp.n[0]) == int(dk.n[0]), "row counts differ"
print("averages match:", abs(float(sp.avg_dep_delay[0]) - float(dk.avg_dep_delay[0])) < 1e-6)

Spark : {'n': 7079061.0, 'avg_dep_delay': 12.677046863546439}
DuckDB: {'n': 7079061.0, 'avg_dep_delay': 12.677046863546439}
averages match: True


**(d) Is the flight key unique? And what happens to flights with no tail number?**
Slide 5 lists the key `Tail_Number + FlightDate + flight number + Origin`. A key is only a key if no two rows share it. Also counts how many flights cannot enter a delay chain, and why.

In [26]:
# (d) key uniqueness: rows that share the slide-5 key
KEY = ["Tail_Number", "FlightDate", "Flight_Number_Reporting_Airline", "Origin"]
KEY2 = KEY + ["CRSDepTime"]                      # add scheduled departure time as a tie-breaker
n = fu.count()
d1 = fu.groupBy(*KEY).count().filter("count > 1")
d2 = fu.groupBy(*KEY2).count().filter("count > 1")
print(f"rows: {n:,}")
print(f"key groups with >1 row, slide-5 key        : {d1.count():,}")
print(f"key groups with >1 row, + CRSDepTime       : {d2.count():,}")
print("worst offenders (slide-5 key):")
d1.orderBy(F.desc("count")).show(5, truncate=False)

rows: 105,631,230


key groups with >1 row, slide-5 key        : 704


key groups with >1 row, + CRSDepTime       : 26
worst offenders (slide-5 key):


+-----------+----------+-------------------------------+------+-----+
|Tail_Number|FlightDate|Flight_Number_Reporting_Airline|Origin|count|
+-----------+----------+-------------------------------+------+-----+
|NULL       |2021-03-14|771                            |DEN   |3    |
|NULL       |2020-03-20|770                            |LAS   |2    |
|NULL       |2020-03-27|386                            |DEN   |2    |
|NULL       |2020-03-30|1861                           |LAS   |2    |
|NULL       |2020-03-30|2136                           |BOS   |2    |
+-----------+----------+-------------------------------+------+-----+
only showing top 5 rows


In [27]:
# (e) which flights cannot enter a chain? (same filters as notebook 03, counted one by one)
tail_missing = F.col("Tail_Number").isNull() | (F.col("Tail_Number") == "") | (F.col("Tail_Number") == "UNKNOWN")
r = fu.agg(
    F.count("*").alias("all_flights"),
    F.sum((tail_missing).cast("int")).alias("missing_or_unknown_tail"),
    F.sum(((F.col("Cancelled") == 1) | (F.col("Diverted") == 1)).cast("int")).alias("cancelled_or_diverted"),
    F.sum((tail_missing & (F.col("Cancelled") == 0) & (F.col("Diverted") == 0)).cast("int")).alias("missing_tail_but_flown"),
).toPandas().iloc[0]
for k, v in r.items():
    print(f"{k:26s}{int(v):>13,}  {100*v/r.all_flights:6.2f}%")
print("-> flights with no usable tail number are dropped before chaining; they never appear in a chain or cascade.")

all_flights                 105,631,230  100.00%
missing_or_unknown_tail         443,965    0.42%
cancelled_or_diverted         2,233,448    2.11%
missing_tail_but_flown                0    0.00%
-> flights with no usable tail number are dropped before chaining; they never appear in a chain or cascade.


In [28]:
# (f) are the duplicates only flights with no tail number? Re-test the key on flights that have one.
has_tail = ~tail_missing
a = fu.filter(has_tail).groupBy(*KEY).count().filter("count > 1").count()
b = fu.filter(has_tail).groupBy(*KEY2).count().filter("count > 1").count()
c = fu.filter(tail_missing).groupBy(*KEY2).count().filter("count > 1").count()
print(f"flights WITH a tail number: duplicate groups, slide-5 key      : {a:,}")
print(f"flights WITH a tail number: duplicate groups, + CRSDepTime     : {b:,}")
print(f"flights WITHOUT a tail number: duplicate groups, + CRSDepTime  : {c:,}")

flights WITH a tail number: duplicate groups, slide-5 key      : 15
flights WITH a tail number: duplicate groups, + CRSDepTime     : 14
flights WITHOUT a tail number: duplicate groups, + CRSDepTime  : 12


26/11/11 14:22:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 305541 ms exceeds timeout 120000 ms
26/11/11 14:22:48 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/11 14:22:51 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:841

## 5 · Write the result

In [25]:
(fu.write.mode("overwrite").partitionBy("Year").parquet(str(OUT)))
back = spark.read.parquet(str(OUT))
print(f"wrote {back.count():,} rows to {OUT}")
print(f"size on disk: {sum(p.stat().st_size for p in OUT.rglob('*.parquet'))/1e9:.2f} GB")

wrote 105,631,230 rows to /Users/a../Desktop/ADSP32030_Scalable/Project/data/flights_utc
size on disk: 2.69 GB
